# 02 — Обучение модели
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

> 🕐 Обновлён: 2026-04-10 00:05 МСК

**Время:** ~10–15 минут на T4  
Загружаем из кэша `/content/data/cache.npy` (мгновенно), обучаем LaneCNN,  
сохраняем `/content/models/model_v2.pth` + копия на Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

# Создаём нужные папки на Drive если их нет
os.makedirs('outputs', exist_ok=True)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.dataset_v2 import load_and_cache, get_loaders
from src.model_v2   import build_model
from src.train_v2   import train, evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Загружаем кэш — если нет, пересоздаём из датасета на Drive
import glob, pathlib, pandas as pd

CACHE_PATH = '/content/data/cache.npy'
LOCAL_DATA  = '/content/data'

if not pathlib.Path(CACHE_PATH).exists():
    print("Кэш не найден — распаковываю датасет и пересоздаю...")
    import zipfile, os

    ZIP_PATH = '/content/drive/MyDrive/diploma/data/dataset.zip'
    os.makedirs(LOCAL_DATA, exist_ok=True)

    if not pathlib.Path(f'{LOCAL_DATA}/IMG').exists():
        # Ищем IMG в уже распакованных подпапках
        img_dirs = glob.glob(f'{LOCAL_DATA}/**/IMG', recursive=True)
        if not img_dirs:
            print("Распаковываю zip...")
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                z.extractall(LOCAL_DATA)
            print("Готово")
        else:
            print(f"IMG уже есть: {img_dirs[0]}")

    # Ищем CSV
    csvs = glob.glob(f'{LOCAL_DATA}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError("CSV не найден в /content/data/ — проверь распаковку")
    CSV_PATH = csvs[0]
    print(f"CSV: {CSV_PATH}")

    # Патчим Windows-пути
    _df = pd.read_csv(CSV_PATH)
    for _col in _df.columns[:3]:
        if _df[_col].dtype == object:
            _sample = str(_df[_col].iloc[0]).strip()
            if '\\' in _sample:
                _df[_col] = _df[_col].apply(
                    lambda x: 'IMG/' + str(x).strip().replace('\\', '/').split('/')[-1]
                    if pd.notna(x) else x
                )
                _fixed = str(pathlib.Path(CSV_PATH).parent) + '/driving_log_fixed.csv'
                _df.to_csv(_fixed, index=False)
                CSV_PATH = _fixed
                print(f"Пути исправлены → {CSV_PATH}")
                break

    IMAGES_DIR = str(pathlib.Path(CSV_PATH).parent) + '/'
    images, angles = load_and_cache(CSV_PATH, IMAGES_DIR, CACHE_PATH)
else:
    images, angles = load_and_cache(None, None, CACHE_PATH)

print(f'Загружено: {len(images)} сэмплов')

In [ ]:
# DataLoader: batch_size=128, num_workers=2, pin_memory=True
train_loader, val_loader, test_loader = get_loaders(
    images, angles, batch_size=128
)
print(f'Train батчей: {len(train_loader)}  '
      f'Val: {len(val_loader)}  Test: {len(test_loader)}')

In [ ]:
# Строим модель
model = build_model(DEVICE)

In [ ]:
# Патч train_v2.py — убираем verbose (удалён в PyTorch 2.2+)
import re
_path = '/content/drive/MyDrive/diploma/src/train_v2.py'
_txt  = open(_path).read()
_fixed = _txt.replace(
    'ReduceLROnPlateau(optimizer, factor=0.5, patience=3, verbose=True)',
    'ReduceLROnPlateau(optimizer, factor=0.5, patience=3)'
)
if _fixed != _txt:
    open(_path, 'w').write(_fixed)
    print('train_v2.py исправлен')
else:
    print('train_v2.py уже актуален')

# Перезагружаем модуль чтобы подхватить изменение
import importlib, src.train_v2
importlib.reload(src.train_v2)
from src.train_v2 import train, evaluate

In [ ]:
# Обучаем (20 эпох, EarlyStopping patience=5)
# Модель сохраняется локально — быстро
import os
os.makedirs('/content/models', exist_ok=True)

history = train(
    model, train_loader, val_loader,
    epochs=20,
    lr=3e-4,
    patience=5,
    model_path='/content/models/model_v2.pth',
    device=DEVICE
)

In [ ]:
# ── Кривые обучения ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train loss')
ax.plot(history['val_loss'],   label='Val loss')
ax.set_xlabel('Эпоха')
ax.set_ylabel('MSE Loss')
ax.set_title('Кривые обучения LaneCNN')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Метрики на тестовой выборке ───────────────────────────────────
model.load_state_dict(torch.load('/content/models/model_v2.pth', map_location=DEVICE))
metrics = evaluate(model, test_loader, DEVICE)

print('\n── Метрики на тесте ──────────────────')
print(f'  MSE:      {metrics["mse"]:.5f}')
print(f'  MAE:      {metrics["mae"]:.5f}')
print(f'  RMSE:     {metrics["rmse"]:.5f}')
print(f'  Accuracy: {metrics["accuracy"]*100:.1f}%  (в/вне полосы)')

# Копируем модель на Drive для сохранности
import shutil
os.makedirs('models', exist_ok=True)
shutil.copy('/content/models/model_v2.pth', 'models/model_v2.pth')
print('Модель скопирована на Drive: models/model_v2.pth')

In [ ]:
# ── Scatter: предсказание vs реальное значение ────────────────────
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for imgs, ang in test_loader:
        pred = model(imgs.to(DEVICE)).cpu().numpy().flatten()
        all_pred.extend(pred)
        all_true.extend(ang.numpy().flatten())

all_pred = np.array(all_pred)
all_true = np.array(all_true)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_true, all_pred, alpha=0.3, s=10)
ax.plot([-1, 1], [-1, 1], 'r--', linewidth=1.5, label='Идеал')
ax.axvline(-0.15, color='gray', linestyle=':', alpha=0.7)
ax.axvline( 0.15, color='gray', linestyle=':', alpha=0.7, label='Граница ±0.15')
ax.set_xlabel('Реальный угол')
ax.set_ylabel('Предсказанный угол')
ax.set_title('Предсказание vs Реальное')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/scatter.png', dpi=120)
plt.show()